## Scene Cropping: Evaluation of PointNet++ MSG


In [1]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import plotly.graph_objects as go
import plotly.express as px

# Verify
print(f"PyTorch       : {torch.__version__}")
import plotly
print(f"Plotly        : {plotly.__version__}")

# Device (MPS on Mac)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device        : {device}")

# Paths
_cwd = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == "notebooks" else _cwd

DATA_DIR_M40    = os.path.join(PROJECT_ROOT, "data", "ModelNet40")
CACHE_DIR_M40   = os.path.join(PROJECT_ROOT, "data", "cache_modelnet40_1024pts")
CHECKPOINT_DIR  = os.path.join(PROJECT_ROOT, "checkpoints")
S3DIS_DIR       = os.path.join(PROJECT_ROOT, "data", "S3DIS")

# MSG checkpoint
msg_ckpt_path = os.path.join(CHECKPOINT_DIR, "pointnetpp_msg_best.pt")
assert os.path.exists(msg_ckpt_path), f"MSG checkpoint missing at {msg_ckpt_path}"
print(f"\nMSG checkpoint found: {msg_ckpt_path}")
print(f"  Size: {os.path.getsize(msg_ckpt_path)/1024**2:.1f} MB")

# ModelNet40 classes (what our model knows)
CLASSES_M40 = [
    "airplane", "bathtub", "bed", "bench", "bookshelf", "bottle", "bowl",
    "car", "chair", "cone", "cup", "curtain", "desk", "door", "dresser",
    "flower_pot", "glass_box", "guitar", "keyboard", "lamp", "laptop",
    "mantel", "monitor", "night_stand", "person", "piano", "plant", "radio",
    "range_hood", "sink", "sofa", "stairs", "stool", "table", "tent", "toilet",
    "tv_stand", "vase", "wardrobe", "xbox",
]
CLASS_TO_IDX_M40 = {c: i for i, c in enumerate(CLASSES_M40)}
IDX_TO_CLASS_M40 = {i: c for c, i in CLASS_TO_IDX_M40.items()}

print(f"\nReady. ModelNet40 has {len(CLASSES_M40)} classes.")

PyTorch       : 2.12.0
Plotly        : 6.8.0
Device        : mps

MSG checkpoint found: /Users/dosvatsky/3D Object Detection/checkpoints/pointnetpp_msg_best.pt
  Size: 6.8 MB

Ready. ModelNet40 has 40 classes.


## Defining PointNet++ MSG architecture 

In [2]:
# ===== FPS, Ball Query, index_points =====
def farthest_point_sample(xyz, npoint):
    B, N, _ = xyz.shape
    dev = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    distance = torch.full((B, N), float("inf"), device=dev)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    batch_idx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(npoint):
        centroids[:, i] = farthest
        cxyz = xyz[batch_idx, farthest, :].unsqueeze(1)
        dist = ((xyz - cxyz) ** 2).sum(dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = distance.argmax(dim=-1)
    return centroids


def index_points(points, idx):
    B = points.shape[0]
    vs = list(idx.shape); vs[1:] = [1]*(len(vs)-1)
    rs = list(idx.shape); rs[0] = 1
    bi = torch.arange(B, dtype=torch.long, device=points.device).view(vs).repeat(rs)
    return points[bi, idx, :]


def ball_query(radius, nsample, xyz, new_xyz):
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape
    dev = xyz.device
    gi = torch.arange(N, dtype=torch.long, device=dev).view(1, 1, N).repeat(B, S, 1)
    sd = ((new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2).sum(dim=-1)
    gi[sd > radius ** 2] = N
    gi = gi.sort(dim=-1)[0][:, :, :nsample]
    gf = gi[:, :, 0:1].repeat(1, 1, nsample); gi[gi == N] = gf[gi == N]
    return gi


class SetAbstractionMSG(nn.Module):
    def __init__(self, npoint, radii, nsamples, in_channel, mlps):
        super().__init__()
        self.npoint, self.radii, self.nsamples = npoint, radii, nsamples
        self.conv_blocks, self.bn_blocks = nn.ModuleList(), nn.ModuleList()
        for mlp in mlps:
            convs, bns = nn.ModuleList(), nn.ModuleList()
            last = in_channel + 3
            for c in mlp:
                convs.append(nn.Conv2d(last, c, 1)); bns.append(nn.BatchNorm2d(c)); last = c
            self.conv_blocks.append(convs); self.bn_blocks.append(bns)

    def forward(self, xyz, features=None):
        fps = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps); outs = []
        for i, (r, k) in enumerate(zip(self.radii, self.nsamples)):
            nn_idx = ball_query(r, k, xyz, new_xyz)
            g_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)
            if features is not None:
                g = torch.cat([g_xyz, index_points(features, nn_idx)], dim=-1)
            else:
                g = g_xyz
            g = g.permute(0, 3, 1, 2).contiguous()
            for conv, bn in zip(self.conv_blocks[i], self.bn_blocks[i]):
                g = F.relu(bn(conv(g)))
            outs.append(g.max(dim=-1)[0])
        return new_xyz, torch.cat(outs, dim=1).permute(0, 2, 1).contiguous()


class GlobalSetAbstraction(nn.Module):
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for c in mlp:
            self.convs.append(nn.Conv1d(last, c, 1)); self.bns.append(nn.BatchNorm1d(c)); last = c

    def forward(self, xyz, features):
        x = torch.cat([xyz, features], dim=-1).permute(0, 2, 1)
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x)))
        return x.max(dim=-1)[0]


class PointNetPlusPlusMSG(nn.Module):
    def __init__(self, num_classes=40, dropout=0.5):
        super().__init__()
        self.sa1 = SetAbstractionMSG(512, [0.1, 0.2, 0.4], [16, 32, 128], 0,
                                     [[32, 32, 64], [64, 64, 128], [64, 96, 128]])
        self.sa2 = SetAbstractionMSG(128, [0.2, 0.4, 0.8], [32, 64, 128], 320,
                                     [[64, 64, 128], [128, 128, 256], [128, 128, 256]])
        self.sa_global = GlobalSetAbstraction(640 + 3, [256, 512, 1024])
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  num_classes),
        )

    def forward(self, xyz):
        l1_xyz, l1_f = self.sa1(xyz, None)
        l2_xyz, l2_f = self.sa2(l1_xyz, l1_f)
        return self.classifier(self.sa_global(l2_xyz, l2_f))


print("PointNet++ MSG architecture defined.")

PointNet++ MSG architecture defined.


## Loading MSG checkpoint and doing sanity check on one ModelNet40 sample

In [3]:
# Load the trained checkpoint
model = PointNetPlusPlusMSG(num_classes=40, dropout=0.5).to(device)
ckpt = torch.load(msg_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Loaded checkpoint:")
print(f"  Epoch     : {ckpt.get('epoch', '?')}")
print(f"  Val acc   : {ckpt.get('val_acc', '?')}")
print(f"  Weights   : {'EMA' if ckpt.get('is_ema') else 'raw'}")

# Sanity check: run inference on one chair from ModelNet40 cache
# (this confirms the model loads correctly before we feed it real-scene crops)
cache_path = os.path.join(CACHE_DIR_M40, "test.npz")
if os.path.exists(cache_path):
    data = np.load(cache_path)
    points, labels = data["points"], data["labels"]
    chair_idx = int(np.where(labels == CLASS_TO_IDX_M40["chair"])[0][0])
    pts = torch.from_numpy(points[chair_idx]).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(pts)
        pred = logits.argmax(1).item()
        confidence = F.softmax(logits, dim=1).max().item()
    print(f"\nSanity check on a clean ModelNet40 chair:")
    print(f"  True class : chair  (idx {CLASS_TO_IDX_M40['chair']})")
    print(f"  Predicted  : {IDX_TO_CLASS_M40[pred]}  (idx {pred})")
    print(f"  Confidence : {confidence:.3f}")
    print(f"  Correct?   : {pred == CLASS_TO_IDX_M40['chair']}")
else:
    print(f"\nNo ModelNet40 cache at {cache_path} — skipping sanity check.")

Loaded checkpoint:
  Epoch     : 28
  Val acc   : 0.8987034035656402
  Weights   : EMA

Sanity check on a clean ModelNet40 chair:
  True class : chair  (idx 8)
  Predicted  : chair  (idx 8)
  Confidence : 0.936
  Correct?   : True
